## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import networkx as nx

from fhl_snn import data as dta, complexes as cx, operators as ops
from fhl_snn import experiments as ex
from fhl_snn.utils import Results, markdown_table

RESULTS = Results('../results/all.json')   
DATASET = 'Cora'                            
SEEDS   = (0, 1, 2)                       

In [ ]:
dataset = dta.load_dataset(DATASET, root='../data')
datasets = {DATASET: dataset}
print(dataset)

In [ ]:
rows = ex.exp_preprocessing_cost(datasets, repeat=3)
print(markdown_table(
    [[r['dataset'], r['n_nodes'], r['n_edges'], r['n_triangles'],
      f"{r['t_complex_s']:.3f}", f"{r['t_laplacian_s']:.3f}", f"{r['t_total_s']:.3f}"]
     for r in rows],
    ['dataset', 'nodes', 'edges', 'triangles', 'complex (s)', 'Laplacians (s)', 'total (s)']))

In [ ]:
rows = ex.exp_hypothesis_check(datasets, N=[1, 2])
print(markdown_table(
    [[r['dataset'], r['N'], r['n_edges'], r['n_triangles'],
      f"{r['mean_degree']:.2f}", f"{r['frac_exactly_2']:.2f}",
      'YES' if r['satisfied'] else 'NO'] for r in rows],
    ['dataset', 'N', 'edges', 'triangles', 'mean degree', 'frac == 2', 'satisfied']))

In [ ]:
graphs = {
    'cycle C_20 (hypothesis holds)': dta.synthetic_graph('cycle', n=20),
    'karate (hypothesis fails)':     nx.karate_club_graph(),
    'G(20, 0.3)':                    dta.synthetic_graph('sbm', n=20, blocks=2, p_in=.4, p_out=.2),
}
rows = ex.exp_operator_wellposedness(graphs)
print(markdown_table(
    [[r['graph'][:28], r['operator'], 'YES' if r['psd'] else 'NO',
      f"{r['lambda_min']:+.3f}", 'YES' if r['m_matrix'] else 'NO',
      r['n_pos_offdiag'], f"{r['norm_L1']:.2f}",
      'YES' if r['lambda_max_bound_ok'] else f"NO ({r['violation_factor']:.1f}x)"]
     for r in rows],
    ['graph', 'operator', 'PSD', 'min eig', 'M-matrix', '#pos offdiag',
     '||L.1||', 'lmax <= N+1']))

In [ ]:
rows = ex.exp_eq4_normalisation(graphs, gamma=0.5, p=0.5, N=0)
print(markdown_table(
    [[r['graph'][:28], f"{r['lambda_max']:.2f}", f"{r['fixed_min_entry']:+.3f}",
      'YES' if r['fixed_stochastic'] else 'NO', f"{r['matched_min_entry']:+.3f}",
      'YES' if r['matched_stochastic'] else 'NO'] for r in rows],
    ['graph', 'lambda_max', 'Eq.(4) min entry', 'Eq.(4) stoch.',
     'matched min entry', 'matched stoch.']))

In [ ]:
G = dta.synthetic_graph('cycle', n=20)
L_eq3 = cx.eq3_down_laplacian(sorted(tuple(sorted(e)) for e in G.edges()))

rows = ex.exp_chebyshev_properties(L_eq3, gammas=(0.3, 0.5), Ks=(10, 30, 60))
print(markdown_table(
    [['shifted' if r['shift'] else 'raw', r['gamma'], r['K'],
      f"{r['min_entry']:+.2e}", r['n_neg_offdiag'],
      f"{r['max_rowsum_dev']:.2e}", f"{r['max_colsum_dev']:.2e}",
      f"{r['row_col_gap']:.1e}", f"{r['norm_inf']:.4f}"] for r in rows],
    ['filter', 'gamma', 'K', 'min entry', '#neg offdiag', 'row-sum dev',
     'col-sum dev', '|row-col|', '||P||_inf']))

In [ ]:
rows = ex.exp_chebyshev_error(L_eq3, gammas=(0.3, 0.5, 0.7), Ks=(10, 20, 40, 80))
print(markdown_table(
    [[r['gamma'], f"{r['loglog_slope']:.2f}", f"{r['predicted_slope']:.2f}",
      f"{r['loglog_r2']:.4f}", f"{r['semilog_r2']:.4f}",
      ', '.join(f'{e:.1e}' for e in r['errors'])] for r in rows],
    ['gamma', 'log-log slope', 'predicted -2g', 'log-log R^2', 'semilog R^2',
     'errors (K=10,20,40,80)']))

In [ ]:
rows = ex.exp_runtime_comparison(n=1000, gamma=0.5, Ks=(20, 50), feature_dims=(1, 64))
print(markdown_table(
    [[r['method'], r['operand'], f"{r['seconds']:.4f}", f"{r['speedup']:.1f}x"]
     for r in rows],
    ['method', 'operand', 'seconds', 'speedup']))

In [ ]:
rows = ex.exp_k_study(dataset, Ks=(4, 8, 16, 32), gamma=0.5, seeds=SEEDS,
                      epochs=80, store=RESULTS, tag=DATASET)
print(markdown_table(
    [[r['K'], f"{r['deficit']:.4f}", r['auc_str'], f"{r['seconds']:.0f} s"] for r in rows],
    ['K', 'row-sum deficit', 'test ROC AUC', 'time/run']))

In [ ]:
rows = ex.exp_gamma_sweep(dataset, gammas=(0.1, 0.3, 0.5, 0.7, 0.9, 1.0),
                          seeds=SEEDS, K=16, epochs=120, store=RESULTS, tag=DATASET)
print(markdown_table([[r['gamma'], r['auc_str'], r['ap_str']] for r in rows],
                     ['gamma', 'test ROC AUC', 'test AP']))

base = next(r for r in rows if r['gamma'] == 1.0)['auc_mean']
best = max((r for r in rows if r['gamma'] < 1.0), key=lambda r: r['auc_mean'])
print(f"\nnon-fractional (gamma=1.0): {base*100:.2f}")
print(f"best fractional (gamma={best['gamma']}): {best['auc_mean']*100:.2f}")
print(f"difference: {(best['auc_mean']-base)*100:+.2f} AUC points")

In [ ]:
rows_node = ex.exp_gamma_sweep(dataset, gammas=(0.3, 0.5, 1.0), seeds=SEEDS, K=16,
                               use_simplicial=False, epochs=120,
                               store=RESULTS, tag=DATASET + '_nodeonly')
print(markdown_table([[r['gamma'], r['auc_str']] for r in rows_node],
                     ['gamma', 'test ROC AUC (node pathway only)']))

In [ ]:
rows = ex.exp_expander_mixing()
print(markdown_table(
    [[r['graph'][:26], f"{r['lambda_2']:.3f}", 'OK' if r['hypothesis'] else 'FAIL',
      r['gamma'], r['t_mix'] if r['t_mix'] else 'n/a',
      f"{r['measured_speedup']:.2f}" if r['measured_speedup'] else '-',
      f"{r['predicted']:.2f}", f"{r['predicted_1']:.2f}"]
     for r in rows],
    ['graph', 'lambda_2', 'lam2<1', 'gamma', 't_mix', 'measured',
     ' ppred.', 'pred.']))

In [ ]:
import numpy as np
pap, cor = [], []
for r in rows:
    if r['measured_speedup'] and r['gamma'] != 1.0:
        pap.append(r['measured_speedup'] / r['predicted'])
        cor.append(r['measured_speedup'] / r['predicted_test'])
print(f"formula     : median measured/predicted = {np.median(pap):.2f}")
print(f"new_formula : median measured/predicted = {np.median(cor):.2f}")
print("(1.00 = exact agreement)")

In [ ]:
n = int(dataset.x.size(0))
edges = dta.undirected_edges(dataset.edge_index)
splits = dta.split_edges(edges, n, seed=0)
A = np.zeros((n, n))
for i, j in splits['train_pos']:
    A[int(i), int(j)] = A[int(j), int(i)] = 1.0
L0 = np.diag(A.sum(1)) - A

rows = ex.exp_operator_density(L0, gammas=(0.5,))
print(markdown_table(
    [[r['operator'], r['nnz_offdiag'], f"{r['density']:.4f}",
      f"{r.get('ratio', 1):.1f}x"] for r in rows],
    ['operator', 'off-diag non-zeros', 'density', 'ratio vs L']))

In [ ]:
rows, dens = ex.exp_structure_control(dataset, gamma=0.5, seeds=SEEDS, epochs=80)
print()
print(markdown_table([[r['operator'], r['auc_str']] for r in rows],
                     ['operator', 'test ROC AUC']))